In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)


import numpy as np 
import torch
from torchvision import datasets, transforms
import torch.nn.functional as F


import pandas as pd
import os
from local_models_piece import CNN, Mnist_relu_4_1024, AlibiCNN, LeNet5, ResNet, MLP_schut, AE, Generator, BasicBlock
from collections import OrderedDict
from matplotlib import pyplot as plt
from skimage.metrics import structural_similarity
from joblib import Parallel, delayed


In [ ]:
def compute_ssim(image, counterfactual):
    return structural_similarity(image, counterfactual, data_range=2, multichannel=True)


def get_implausibility(counterfactual, subset, y_train, target, new_pred, method):
    if counterfactual is None:
        return None
    
    else:
        if method == "Captum-MinParamPerturbation":
            subset = subset[y_train == new_pred]
        else:
            subset = subset[y_train == target]
        
        # Flatten the images in the subset and the counterfactual
        flattened_subset = np.stack([x.ravel() for x in subset])  # Shape: (num_images, flattened_image_size)
        flattened_counterfactual = counterfactual.ravel()  # Shape: (flattened_image_size,)
        
        #ssim_scores_list = []
        # Compute ssim score between each subset and the explanation
        #for i in range(flattened_subset.shape[0]):
        #    ssim_score = structural_similarity(flattened_subset[i], flattened_counterfactual, data_range=2, multichannel=True)
        #    ssim_scores_list.append(ssim_score)

        ssim_scores_list = Parallel(n_jobs=-1)(delayed(compute_ssim)(flattened_subset[i], flattened_counterfactual)
                                       for i in range(flattened_subset.shape[0]))

   
        
        # Compute the mean squared distance (implausibility)
        #print(np.mean(ssim_scores_list))
        return np.mean(ssim_scores_list)


def get_unfaithfulness(counterfactual, subset, y_train, target, new_pred, method):
    # The only difference here is that subset should be a set of gerenerated images using SGLD 
    # Im not sure how to generate these instances with SGLD and if this is even a 'faithful' way to capture faithfulness
    if counterfactual is None:
        return None
    
    else:
        if method == "Captum-MinParamPerturbation":
            subset = subset[y_train == new_pred]
        else:
            subset = subset[y_train == target]
        
        # Flatten the images in the subset and the counterfactual
        flattened_subset = np.stack([x.ravel() for x in subset])  # Shape: (num_images, flattened_image_size)
        flattened_counterfactual = counterfactual.ravel()  # Shape: (flattened_image_size,)
        
        # Compute squared L2 distances in a vectorized way
        differences = flattened_subset - flattened_counterfactual  # Broadcasting
        squared_distances = np.sum(differences ** 2, axis=1)
        
        # Compute the mean squared distance (implausibility)
        return np.mean(squared_distances)

def get_l1(counterfactual, original):
    if counterfactual is None:
        return None
    else:
        return np.sum(np.abs(original - counterfactual))

def get_l2(counterfactual, original):
    if counterfactual is None:
        return None
    else:
        return np.linalg.norm(original- counterfactual)

def get_linf(counterfactual,original):
    if counterfactual is None:
        return None
    else:
        return np.max(np.abs(original - counterfactual))

def get_correctness(counterfactual, target, network):

    if network(counterfactual) == target:
        return True
    return False

def correctness(row):
    if row['Target Label'] == row['New Prediction']: 
        val = 1
    else:
        val = 0
    return val


def weak_correctness(row):
    #### Change code #### --> at least 1 explanation/decision boundary for that instance is correct
    if row['Target Label'] == row['New Prediction']: 
        val = 1
    else:
        val = 0
    return val


def get_IM1(counterfactual, aes, og_prediction, new_prediction):
    """
    return: IM1 metric
    """
    if counterfactual is None:
        return None
    else:
        explanation_img = (counterfactual * 0.5) + 0.5
        explanation_img = torch.from_numpy(explanation_img).float() 
        o_recon = aes[og_prediction](explanation_img.reshape(-1,1,28,28)).flatten()
        e_recon = aes[int(new_prediction)](explanation_img.reshape(-1,1,28,28)).flatten()
        o_error = sum(   (  o_recon.detach().numpy().flatten() - explanation_img.detach().numpy().flatten() )**2   )  
        e_error = sum(   (  e_recon.detach().numpy().flatten() - explanation_img.detach().numpy().flatten() )**2   )  
        return e_error / o_error

def get_IM2(counterfactual, aes, aefull, new_prediction):
    """
    return: IM2 metric
    """
    if counterfactual is None:
        return None
    else:
        
        explanation_img = (counterfactual * 0.5) + 0.5
        explanation_img = torch.from_numpy(explanation_img).float() 
        all_recon = aefull(explanation_img.reshape(-1,1,28,28)).flatten().detach().numpy()
        e_recon = aes[int(new_prediction)](explanation_img.reshape(-1,1,28,28)).flatten().detach().numpy()
        x_l1_norm = float(sum(abs(explanation_img.flatten())))
        return sum(  (e_recon - all_recon)**2  ) / x_l1_norm


def get_prediction(image, model, device = 'cpu'):
    if isinstance(image, np.ndarray):
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
        image = transform(image).unsqueeze(0)  # Convert to tensor and add batch dimension
    
    image = image.to(device, dtype=torch.float32)
    model = model.to(device)

    # Get the model's prediction
    with torch.no_grad():  # No gradients needed for inference
        output = model(image)[0].argmax()  
    
    return int(output) 





def compute_morans_i(image, kernel_name, size):
        """
        TODO
        :param info:
        :return:
        """
        
        if image is None:
            return None
        else:
            if isinstance(image, torch.Tensor):
                pass
            else:
                image = torch.from_numpy(image)

            
            ROOK_KERNEL = torch.Tensor([
                                    [0, 1, 0],
                                    [1, 0, 1],
                                    [0, 1, 0],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            BISHOP_KERNEL = torch.Tensor([
                                    [1, 0, 1],
                                    [0, 0, 0],
                                    [1, 0, 1],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            QUEEN_KERNEL = torch.Tensor([
                                    [1, 1, 1],
                                    [1, 0, 1],
                                    [1, 1, 1],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            VERT_KERNEL = torch.Tensor([
                                    [0, 1, 0],
                                    [0, 0, 0],
                                    [0, 1, 0],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            HOR_KERNEL = torch.Tensor([
                                    [0, 0, 0],
                                    [1, 0, 1],
                                    [0, 0, 0],
                                    ]).unsqueeze(0).unsqueeze(0).requires_grad_(False)

            KERNEL_BY_NAME = {'rook': ROOK_KERNEL,'bishop': BISHOP_KERNEL,'queen': QUEEN_KERNEL,'vert': VERT_KERNEL,'hor': HOR_KERNEL}
        
            # Make sure the chosen kernel is of shape (input_channels, output_channels, width, height) = (1, 1, 28, 28)
            device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
            kernel = KERNEL_BY_NAME[kernel_name].to(device)
            assert kernel.shape == (1, 1, 3, 3)
            

            image_shape = (size, size)  #  make variable. currently hardcoded for mnist.

            # Compute W
            num_neighbours = F.conv2d(torch.ones((1, 1) + image_shape, device=device),
                                    kernel,
                                    padding=1).sum()

            # Normalize the nodes' weights and reshape them to an image
            # Unsqueeze to make a channel dimension (with only one channel)
            # Unsqueeze again to make a minibatch dimension (with only one element)
            norm_weights = (image - torch.mean(image)).reshape(image_shape).unsqueeze(0).unsqueeze(0) 
            
            # Compute the numerator of the fraction
        
            numerator = (F.conv2d(norm_weights,kernel,padding=1) * norm_weights).sum()
            numerator *= (size*size)
            # Compute the denominator of the fraction
            denominator = (norm_weights * norm_weights).sum()
            denominator *= num_neighbours
            # Obtain Moran's I
            I = numerator / denominator
            #print("Moran's I is: ", I.item())
            # Value generally falls between [-1, 1]. Rescale to proper regularization term
            #morans_i = torch.abs(I - 1)

            #morans_loss = morans_i_strength * morans_i #this term can be added to the loss function (with a certain strength) such that the network is optimized to have a high autocorrelation value. 

            return I.item()



In [ ]:
def load_weird_relu(model, path):
        model = Mnist_relu_4_1024()
        print(model)
        checkpoint = torch.load(path, map_location='cpu')

        #remove constant.value
        del checkpoint['Constant.value']
        new_state_dict = OrderedDict()
        for key, value in checkpoint.items():
            if key.startswith('Gemm.'):
                new_key = key.replace('Gemm', 'layer1')

            elif key.startswith('Gemm_1'):
                new_key = key.replace('Gemm_1', 'layer2')

            elif key.startswith('Gemm_2'):
                new_key = key.replace('Gemm_2', 'layer3')

            elif key.startswith('Gemm_3'):
                new_key = key.replace('Gemm_3', 'layer4')

            else:
                pass

            new_state_dict[new_key] = value
        checkpoint = new_state_dict
        model.load_state_dict(checkpoint)
        return model

def get_mnist():
    # Define the transform to convert images to tensors and normalize to [-1, 1]
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Download and load the training and test datasets with transformations
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

    # Extract transformed data and labels by iterating over the datasets
    x_train = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))]).numpy()
    y_train = torch.tensor([train_dataset[i][1] for i in range(len(train_dataset))]).numpy()
    x_test = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))]).numpy()
    y_test = torch.tensor([test_dataset[i][1] for i in range(len(test_dataset))]).numpy()

    return x_train, y_train, x_test, y_test

    
def load_image(image_path):
    return torch.load(image_path) # Assuming images are stored as torch tensors


def load_model(model_path, name):
    if name == Mnist_relu_4_1024:
        model = load_weird_relu(name, model_path)
    elif name == ResNet:
         model = name(BasicBlock, [1, 1, 1], num_classes=10)
         model.load_state_dict(torch.load(model_path))
    else:
        model = name()
        model.load_state_dict(torch.load(model_path))

    model.eval()
    return model
    
def load_autoencoders():
	aes = list()
	for i in range(10):
		model = AE()
		model.load_state_dict(torch.load('AAAI-2021-semifactual/AAAI-2021-master/weights/ae_' + str(i) + '.pth', map_location='cpu'))
		model.eval()
		aes.append(model)
	ae_full = AE()
	ae_full.load_state_dict(torch.load('AAAI-2021-semifactual/AAAI-2021-master/weights/ae_full.pth', map_location='cpu'))
	ae_full.eval()
	return aes, ae_full





In [ ]:
network_list = ["mnist_output_100", "mnist_cnn_output_100","mnist_alibi_cnn_output","mnist_lenet5_output", "mnist_resnet8_output", "mnist_schut_mlp_output"]
base_models = "models/mnist"
method_list = ["alibi-CF", "alibi-Proto-CF", "C-Min-Edit", "Captum-MinParamPerturbation", "Min-Edit", "PIECE"]

network_loader = {
    "mnist_cnn_output_100":[CNN, "models/mnist/mnist_cnn_6_128.pt"],
 "mnist_output_100": [Mnist_relu_4_1024, "models/mnist/mnist_relu_4_1024.pt"],
 "mnist_alibi_cnn_output": [AlibiCNN, "models/mnist/mnist_alibi_cnn.pt"],
 "mnist_lenet5_output": [LeNet5, "models/mnist/mnist_lenet5.pt"],
 "mnist_schut_mlp_output": [MLP_schut, "models/mnist/mnist_schut_mlp.pt"],
 "mnist_resnet8_output": [ResNet, "models/mnist/mnist_resnet8.pt"],

}

captum = "output_Captum_mnist_data.csv"
alibi = "output_Alibi_mnist_data.csv"
piece = "output_PIECE_mnist_data.csv"

# Loading AEs for IM1 and IM2 metrics
aes, ae_full = load_autoencoders()

df = pd.DataFrame(columns=["network", "image", "original_label", "original_predicted_label", "predicted_label", "target", "method", "IM1", "IM2", "correctness", "l1", "l2", "linf", "implausibility", "morans_i_original", "morans_i_explanation", "morans_i_original_minus_explanation", "optim_time"])
x_train, y_train, x_test, y_test  = get_mnist()

for network in network_list:
    print("-------------",network, "-------------")
    network_path = os.path.join(base_models, network).replace("\\", "/")
    net_info = network_loader[network]
    model = load_model(net_info[1], net_info[0])
    captum_csv = pd.read_csv(network + "/" + captum)
    alibi_csv = pd.read_csv(network+ "/" + alibi)
    piece_csv = pd.read_csv(network + "/" + piece)
    # align and rename instances (e.g 12 becomes 11, etc. )
    piece_csv["Instance"] -= 1
    frames = [piece_csv, alibi_csv, captum_csv]
    methods_csv = pd.concat(frames)
    methods_csv.reset_index(drop=True)
    # drop instances where we only have data for some of the methods (a bug in one of the scripts causes this) 
    methods_csv = methods_csv[methods_csv['Instance'] != -1]
    methods_csv = methods_csv[methods_csv['Instance'] != 99]
    methods_csv.reset_index(drop=True)
    methods_csv['correctness'] = methods_csv.apply(correctness, axis=1)

    # Check NaN values --> captum has NaNs in target, which is okay. alibi-... has NaNs if no valid explanation was found.
    rows_with_nan = methods_csv[methods_csv.isnull().any(axis=1)]
    #print(rows_with_nan)

    for method in method_list:
        print("**",method, "**")
        method_path = os.path.join(network, method).replace("\\", "/")
        # Iterate through the methods
        filtered_df = methods_csv[methods_csv['Name'] == method]
        for index, row in filtered_df.iterrows():
            #method_path = os.path.join(network, method).replace("\\", "/")
            i = row['Instance']
            t = row['Target Label']
            p = row['New Prediction']
            y_original = row['Original Label']
            #print("This is instance " + str(int(i)) + " with original label " + str(int(y_original)))


            image_original = torch.load(network +"/original/instance_" + str(int(i)+1) +".pt").detach().numpy()
        

            if method == 'PIECE' or method == 'Min-Edit' or method == 'C-Min-Edit':
                new_i = i + 1
                #print("Loading:", method_path +"/instance_" + str(int(i)) + "_target_"+ str(int(t)) +".pt")
                image = torch.load(method_path + "/instance_" + str(int(new_i)) + "_target_"+ str(int(t)) +".pt").detach().numpy()


            elif method == 'Captum-MinParamPerturbation':
                if pd.isna(p):
                    image = None
                #print("Loading:", method_path +"/instance_" + str(int(i)) + "_target_"+ str(int(p)) +".pt")
                else:
                    image = torch.load(method_path + "/instance_" + str(int(i)) + "_target_"+ str(int(p)) +".pt").detach().numpy()

            else:
                #print("Loading:", method_path +"/instance_" + str(int(i)) + "_target_"+ str(int(t)) +".pt")
                if pd.isna(p):
                    image = None
                else:
                    image = torch.load(method_path +"/instance_" + str(int(i)) + "_target_"+ str(int(t)) +".pt").detach().numpy()
                    #print("min image val:", image.min())
                    

            if image is None:
                image_diff = None
            else:
                image_diff = np.abs(image_original - image)
            # Create a dictionary to store results
            
            temp = {
                "network": network,
                "image": i,
                "original_label": int(row['Original Label']),
                "original_predicted_label": int(row['Original Prediction']),
                "predicted_label": p,
                "target": t,
                "method": method,
                "optim_time": row['optim_time'],
                "IM1": get_IM1(image, aes, int(row['Original Prediction']), p),
                "IM2": get_IM2(image, aes, ae_full, p), 
                "correctness": int(row['correctness']), 
                "l1": get_l1(image, image_original),
                "l2": get_l2(image, image_original),
                "linf": get_linf(image, image_original), 
                "implausibility": get_implausibility(image, x_train, y_train, t, p, method),
                "morans_i_original": compute_morans_i(image_original, "queen", 28), 
                "morans_i_explanation": compute_morans_i(image, "queen", 28),
                "morans_i_original_minus_explanation": compute_morans_i(image_diff, "queen", 28)
                
            }
            df_temp = pd.DataFrame([temp])
            df = pd.concat([df, df_temp], ignore_index=True)
        

#df.to_csv("temp.csv") 
df.head()           
            
  
"""
#f, axarr = plt.subplots(1,2)
#axarr[0].imshow(torch_image.reshape(28,28), interpolation='nearest')
#axarr[1].imshow(original_image.reshape(28,28), interpolation='nearest')

"""



In [ ]:
df['correctness_ratio'] = df.groupby(['network', 'method', 'image'])['correctness'].transform('sum') / 9

In [ ]:
# compute weak correctness 
df['weak_correctness'] = df.groupby(['network', 'method', 'image'])['correctness'].transform('max')

In [ ]:
df[df['method'] == 'PIECE'].head(50)

In [ ]:
# add moran's i metric = |MI(x) - MI(x')|

df['morans_i_difference'] = (df['morans_i_original'] - df['morans_i_explanation']).abs()

In [ ]:
df[df['method'] == 'PIECE'].head()

In [ ]:
import glob
import os
import re

In [ ]:
# Add column to confirm whether or not it timed out (PIECE). Regarding Captum and Alibi methods, if there is a NaN then it timed out. 
# Need .out files for this ... this will be painful ...

timeout_text = "Explanation is invalid! Returning incorrect explanation."





timeout_explanation_count = {
    'network': [],
    'image': [],
    'method': [],
    'target': [],
    'timeout': []
}

next_line = 0

for network in network_list:
    print("-------------",network, "-------------")
    out_file = glob.glob(os.path.join(network, '*.out'))
    print(out_file)
    with open(out_file[0], 'r') as file:
        for line in file:
            if line.startswith(timeout_text):
                next_line = 2
                continue

            elif next_line == 2 and line.startswith("   Instance"):
                next_line = 1
                continue

            elif next_line == 1:
                split_list = line.split()
                #print(split_list)
                timeout_explanation_count['network'].append(network)
                timeout_explanation_count['image'].append(float(split_list[1]))
                timeout_explanation_count['target'].append(float(split_list[4]))
                timeout_explanation_count['method'].append(split_list[2])
                timeout_explanation_count['timeout'].append(int(1))
                next_line = 0
                continue

            else:
                continue


            

            
                


df_timeouts = pd.DataFrame(timeout_explanation_count)
df_timeouts.head(20)


In [ ]:
df

In [ ]:
merged_df = pd.merge(df, df_timeouts, on=['network', 'image', 'method', 'target'], how='left')

In [ ]:
merged_df[merged_df['method'] == 'PIECE']

In [ ]:
merged_df.loc[
    ((merged_df['method'] == 'PIECE') | (merged_df['method'] == 'C-Min-Edit') | (merged_df['method'] == 'Min-Edit')) & (merged_df['timeout'].isna()), 
    'timeout'
] = 0 

In [ ]:
merged_df[merged_df['method'] == 'PIECE']

In [ ]:
merged_df.loc[
    ((merged_df['method'] == 'alibi-CF') | (merged_df['method'] == 'alibi-Proto-CF') | (merged_df['method'] == 'Captum-MinParamPerturbation')) 
    & (merged_df['predicted_label'].isna()), 
    'timeout'
] = 1

In [ ]:
merged_df.loc[
    ((merged_df['method'] == 'alibi-CF') | (merged_df['method'] == 'alibi-Proto-CF') | (merged_df['method'] == 'Captum-MinParamPerturbation')) 
    & (merged_df['predicted_label'].notna()), 
    'timeout'
] = 0

In [ ]:
merged_df[merged_df['method'] == 'Captum-MinParamPerturbation'][merged_df['timeout'] == 1]

In [ ]:
merged_df.to_csv("mnist_full_results_updated.csv") 